# Tower OTC Strategies — Backtest & Exploration

All 5 strategies backtested on 122K candles (8 days, EURUSD-OTC).  
Then: exploring new ideas to push accuracy higher **without skipping too many trades**.

**The tradeoff:** Every filter that increases win rate also skips trades. Skipped trades = lost profit.  
The sweet spot is maximum $/hour, not maximum win rate.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import warnings, os, random
warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Load data
data_files = ['../data/eurusd_otc_all.csv', '../data/eurusd_otc_live.csv', '../data/eurusd_otc_5s.csv']
dfs = []
for f in data_files:
    if os.path.exists(f):
        dfs.append(pd.read_csv(f))
df = pd.concat(dfs, ignore_index=True)
df = df.drop_duplicates(subset='timestamp').sort_values('timestamp').reset_index(drop=True)
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df = df.set_index('datetime').sort_index()

hours = len(df) * 5 / 3600
print(f"Dataset: {len(df):,} candles | {hours:.0f} hours ({hours/24:.1f} days)")
print(f"Range: {df.index[0]} to {df.index[-1]}")

In [ ]:
# Build minute-level trade data (entry at :00, result at next :00 = 60s window)
df['second'] = df.index.second

# Get price at each :00 boundary
candles_00 = df[df['second'] == 0][['close']].copy()
candles_00 = candles_00[~candles_00.index.duplicated(keep='first')]
candles_00.columns = ['entry_price']
candles_00['result_price'] = candles_00['entry_price'].shift(-1)
candles_00['went_up'] = (candles_00['result_price'] > candles_00['entry_price']).astype(int)
candles_00['prev_went_up'] = candles_00['went_up'].shift(1)
candles_00['move'] = abs(candles_00['result_price'] - candles_00['entry_price'])
candles_00['prev_move'] = candles_00['move'].shift(1)
candles_00 = candles_00.dropna()

# Also get :30 entries for v1 comparison
candles_30 = df[df['second'] == 30][['close']].copy()
candles_30 = candles_30[~candles_30.index.duplicated(keep='first')]
candles_30.columns = ['entry_price']
candles_30['result_price'] = candles_30['entry_price'].shift(-1)
# Result is 30s later (at :00), approximate with next :30 entry
# Actually need the :00 price. Let's merge.
prices_00 = df[df['second'] == 0][['close']].copy()
prices_00 = prices_00[~prices_00.index.duplicated(keep='first')]
# For each :30, result is next :00
candles_30['result_price'] = None
for i, idx in enumerate(candles_30.index):
    next_00 = prices_00.index[prices_00.index > idx]
    if len(next_00) > 0:
        candles_30.loc[idx, 'result_price'] = prices_00.loc[next_00[0], 'close']
candles_30['result_price'] = candles_30['result_price'].astype(float)
candles_30['went_up'] = (candles_30['result_price'] > candles_30['entry_price']).astype(int)
candles_30['prev_went_up'] = candles_30['went_up'].shift(1)
candles_30['move'] = abs(candles_30['result_price'] - candles_30['entry_price'])
candles_30['prev_move'] = candles_30['move'].shift(1)
candles_30 = candles_30.dropna()

print(f"Trade windows (entry :00, 60s): {len(candles_00):,}")
print(f"Trade windows (entry :30, 30s): {len(candles_30):,}")
print(f"\nBase momentum (follow last, :00): {(candles_00['went_up'] == candles_00['prev_went_up']).mean():.1%}")
print(f"Base momentum (follow last, :30): {(candles_30['went_up'] == candles_30['prev_went_up']).mean():.1%}")

## 1. Backtest Engine

Shared backtest function with martingale. Used by all strategies below.

In [ ]:
def backtest(trades_df, should_trade_mask=None, payout=0.85, max_losses=8, max_stake=200, base_stake=1.0):
    """Backtest follow-last-result with martingale on a trades DataFrame.
    
    trades_df must have: 'went_up', 'prev_went_up' columns.
    should_trade_mask: boolean Series — True = trade, False = skip (observe only).
    
    Returns dict with full stats and equity curve.
    """
    outcomes = trades_df['went_up'].values
    predictions = trades_df['prev_went_up'].values  # follow last result
    
    if should_trade_mask is None:
        mask = np.ones(len(trades_df), dtype=bool)
    else:
        mask = should_trade_mask.values if hasattr(should_trade_mask, 'values') else np.array(should_trade_mask)
    
    balance = 0.0
    equity = [0.0]
    stake = base_stake
    consec_losses = 0
    wins = 0
    losses = 0
    skips = 0
    busts = 0
    max_drawdown = 0.0
    peak = 0.0
    loss_streaks = []
    current_streak = 0
    
    # Track direction through skips (observe even when not trading)
    last_direction = None
    
    for i in range(len(outcomes)):
        if not mask[i]:
            # Skip — but still observe the result for next trade's direction
            skips += 1
            last_direction = outcomes[i]
            equity.append(balance)
            continue
        
        # Direction: follow last observed result
        if last_direction is None:
            bet = random.randint(0, 1)  # first trade = random
        else:
            bet = last_direction
        
        # Martingale reset
        if consec_losses >= max_losses or stake > max_stake:
            busts += 1
            stake = base_stake
            consec_losses = 0
        
        # Result
        won = (bet == outcomes[i])
        if won:
            balance += stake * payout
            wins += 1
            stake = base_stake
            consec_losses = 0
            if current_streak > 0:
                loss_streaks.append(current_streak)
            current_streak = 0
        else:
            balance -= stake
            losses += 1
            consec_losses += 1
            current_streak += 1
            stake = round((stake + base_stake) / payout, 2)
        
        # Track drawdown
        if balance > peak:
            peak = balance
        dd = peak - balance
        if dd > max_drawdown:
            max_drawdown = dd
        
        last_direction = outcomes[i]
        equity.append(balance)
    
    if current_streak > 0:
        loss_streaks.append(current_streak)
    
    total = wins + losses
    total_all = wins + losses + skips
    hours_traded = total_all / 60  # ~1 trade opportunity per minute
    
    return {
        'wins': wins,
        'losses': losses,
        'skips': skips,
        'total_trades': total,
        'total_opportunities': total_all,
        'win_rate': wins / total if total > 0 else 0,
        'skip_pct': skips / total_all if total_all > 0 else 0,
        'profit': balance,
        'per_trade': balance / total if total > 0 else 0,
        'per_hour': balance / hours_traded if hours_traded > 0 else 0,
        'busts': busts,
        'max_drawdown': max_drawdown,
        'max_loss_streak': max(loss_streaks) if loss_streaks else 0,
        'equity': equity,
    }

def print_result(name, r):
    print(f"  {name:<45} | WR: {r['win_rate']:>5.1%} | "
          f"Skip: {r['skip_pct']:>5.1%} | "
          f"Profit: ${r['profit']:>8.2f} | "
          f"$/hr: ${r['per_hour']:>6.2f} | "
          f"Busts: {r['busts']} | "
          f"MaxDD: ${r['max_drawdown']:.2f} | "
          f"MaxStreak: {r['max_loss_streak']}L")

print("Backtest engine ready.")

## 2. The Five Strategies — Head to Head

| Version | Entry | Filter | Target |
|---------|-------|--------|--------|
| v1 | :30 | None | Original baseline |
| v2 | :00 | None | Max profit |
| v2.5 | :00 | Skip tiny moves (<2 pips) | Real money safety |
| v3 | :00 | Skip tiny + fresh reversals | Higher accuracy |
| v4 | :00 | Large moves only (>5.8 pips) | Maximum safety |

In [ ]:
# Enrich candles_00 with features needed by all strategies
# Momentum age: how many consecutive same-direction results
ages = []
age = 0
prev = None
for d in candles_00['went_up'].values:
    if d == prev:
        age += 1
    else:
        age = 1
    ages.append(age)
    prev = d
candles_00['momentum_age'] = ages
candles_00['prev_momentum_age'] = candles_00['momentum_age'].shift(1)

# Move size quantiles for labeling
candles_00['move_pct'] = candles_00['prev_move'].rank(pct=True)
candles_00 = candles_00.dropna()

print("Features added: momentum_age, prev_momentum_age, move_pct")
print(f"  Move size percentiles: p20={candles_00['prev_move'].quantile(0.2):.6f}, "
      f"p50={candles_00['prev_move'].quantile(0.5):.6f}, "
      f"p80={candles_00['prev_move'].quantile(0.8):.6f}")

In [ ]:
# Run all 5 strategies
results = {}

# v1: entry at :30, no filter
r_v1 = backtest(candles_30)
results['v1 (:30, no filter)'] = r_v1

# v2: entry at :00, no filter
r_v2 = backtest(candles_00)
results['v2 (:00, no filter)'] = r_v2

# v2.5: entry at :00, skip tiny moves
mask_v25 = (candles_00['prev_move'] >= 0.00019) | (candles_00['prev_move'].isna())
# First trade always enters
mask_v25.iloc[0] = True
r_v25 = backtest(candles_00, mask_v25)
results['v2.5 (:00, skip tiny <2pip)'] = r_v25

# v3: entry at :00, skip tiny + fresh reversals (age < 2)
mask_v3 = (candles_00['prev_move'] >= 0.00019) & (candles_00['prev_momentum_age'] >= 2)
mask_v3.iloc[0] = True
r_v3 = backtest(candles_00, mask_v3)
results['v3 (:00, skip tiny+fresh)'] = r_v3

# v4: entry at :00, large moves only
mask_v4 = (candles_00['prev_move'] >= 0.00058)
mask_v4.iloc[0] = True
r_v4 = backtest(candles_00, mask_v4)
results['v4 (:00, large only >5.8pip)'] = r_v4

# Print comparison
print(f"{'Strategy':<45} | {'WR':>5} | {'Skip%':>5} | {'Profit':>10} | {'$/hr':>7} | {'Busts':>5} | {'MaxDD':>7} | {'Streak':>6}")
print("—" * 115)
for name, r in results.items():
    print_result(name, r)

In [ ]:
# Equity curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['#888888', 'lime', 'cyan', 'orange', 'magenta']
for (name, r), color in zip(results.items(), colors):
    lw = 2.5 if 'v2 ' in name or 'v2.5' in name else 1.2
    axes[0].plot(r['equity'], label=f"{name} (${r['profit']:.0f})", color=color, linewidth=lw)
axes[0].set_title('Equity Curves — All Strategies')
axes[0].set_xlabel('Trade Opportunity #')
axes[0].set_ylabel('Cumulative P&L ($)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.2)

# $/hour vs Win Rate scatter
names = list(results.keys())
wrs = [r['win_rate'] for r in results.values()]
dphs = [r['per_hour'] for r in results.values()]
skip_pcts = [r['skip_pct'] for r in results.values()]

for i, (name, wr, dph, skip, color) in enumerate(zip(names, wrs, dphs, skip_pcts, colors)):
    axes[1].scatter(wr * 100, dph, s=200, color=color, zorder=5, edgecolors='white')
    axes[1].annotate(name.split('(')[0].strip(), (wr * 100, dph), 
                     textcoords="offset points", xytext=(10, 5), fontsize=9, color=color)
axes[1].set_title('Win Rate vs $/hour — The Tradeoff')
axes[1].set_xlabel('Win Rate (%)')
axes[1].set_ylabel('$/hour')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 3. Where Do Losses Come From?

Before exploring new ideas, let's understand exactly what causes losses. If we can profile the losing trades, we can build smarter filters.

In [ ]:
# Tag each trade as win/loss for follow-last strategy
candles_00['follow_last_correct'] = (candles_00['went_up'] == candles_00['prev_went_up']).astype(int)

# Win rate by move size bucket
candles_00['move_bucket'] = pd.cut(candles_00['prev_move'], 
    bins=[0, 0.00019, 0.00038, 0.00058, 0.001, 1.0],
    labels=['Tiny (<1.9)', 'Small (1.9-3.8)', 'Medium (3.8-5.8)', 'Large (5.8-10)', 'Huge (>10)'])

# Win rate by momentum age
candles_00['age_bucket'] = pd.cut(candles_00['prev_momentum_age'],
    bins=[0, 1, 2, 3, 5, 10, 100],
    labels=['1 (fresh)', '2', '3', '4-5', '6-10', '11+'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# By move size
by_move = candles_00.groupby('move_bucket')['follow_last_correct'].agg(['mean', 'count'])
colors = ['red' if m < 0.80 else 'yellow' if m < 0.90 else 'lime' for m in by_move['mean']]
bars = axes[0].bar(range(len(by_move)), by_move['mean'] * 100, color=colors)
axes[0].set_xticks(range(len(by_move)))
axes[0].set_xticklabels(by_move.index, rotation=30, ha='right')
axes[0].axhline(y=86.9, color='cyan', linestyle='--', alpha=0.5, label='v2 baseline')
axes[0].axhline(y=54.05, color='yellow', linestyle='--', alpha=0.3, label='break-even')
axes[0].set_title('Win Rate by Previous Move Size (pips × 10⁴)')
axes[0].set_ylabel('Win Rate (%)')
axes[0].set_ylim(50, 100)
axes[0].legend(fontsize=8)
for bar, count in zip(bars, by_move['count']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                f'n={count}', ha='center', va='bottom', fontsize=8, color='white')

# By momentum age
by_age = candles_00.groupby('age_bucket')['follow_last_correct'].agg(['mean', 'count'])
colors = ['red' if m < 0.80 else 'yellow' if m < 0.90 else 'lime' for m in by_age['mean']]
bars = axes[1].bar(range(len(by_age)), by_age['mean'] * 100, color=colors)
axes[1].set_xticks(range(len(by_age)))
axes[1].set_xticklabels(by_age.index)
axes[1].axhline(y=86.9, color='cyan', linestyle='--', alpha=0.5, label='v2 baseline')
axes[1].set_title('Win Rate by Momentum Age (consecutive same-dir)')
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_ylim(50, 100)
axes[1].legend(fontsize=8)
for bar, count in zip(bars, by_age['count']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'n={count}', ha='center', va='bottom', fontsize=8, color='white')

# Loss distribution: what % of losses come from each bucket?
loss_mask = candles_00['follow_last_correct'] == 0
total_losses = loss_mask.sum()
loss_by_move = candles_00[loss_mask].groupby('move_bucket').size() / total_losses * 100
axes[2].bar(range(len(loss_by_move)), loss_by_move.values, color='red', alpha=0.7)
axes[2].set_xticks(range(len(loss_by_move)))
axes[2].set_xticklabels(loss_by_move.index, rotation=30, ha='right')
axes[2].set_title('Where Do Losses Come From? (% of all losses)')
axes[2].set_ylabel('% of total losses')
for i, v in enumerate(loss_by_move.values):
    axes[2].text(i, v + 0.5, f'{v:.0f}%', ha='center', fontsize=10, color='white', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nTotal losses (v2): {total_losses} out of {len(candles_00)} trades")
print(f"Losses from Tiny moves: {candles_00[loss_mask & (candles_00['move_bucket'] == 'Tiny (<1.9)')].shape[0]} "
      f"({candles_00[loss_mask & (candles_00['move_bucket'] == 'Tiny (<1.9)')].shape[0]/total_losses*100:.0f}%)")
print(f"Losses from Age=1: {candles_00[loss_mask & (candles_00['prev_momentum_age'] == 1)].shape[0]} "
      f"({candles_00[loss_mask & (candles_00['prev_momentum_age'] == 1)].shape[0]/total_losses*100:.0f}%)")

In [ ]:
# Heatmap: Win rate by move size × momentum age
# This shows the 2D landscape of where losses cluster
pivot = candles_00.groupby(['move_bucket', 'age_bucket'])['follow_last_correct'].agg(['mean', 'count'])
pivot_wr = pivot['mean'].unstack(fill_value=np.nan)
pivot_n = pivot['count'].unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pivot_wr.values * 100, cmap='RdYlGn', vmin=50, vmax=100, aspect='auto')
ax.set_xticks(range(len(pivot_wr.columns)))
ax.set_xticklabels(pivot_wr.columns)
ax.set_yticks(range(len(pivot_wr.index)))
ax.set_yticklabels(pivot_wr.index)
ax.set_xlabel('Momentum Age')
ax.set_ylabel('Previous Move Size')
ax.set_title('Win Rate Heatmap: Move Size × Momentum Age')

for i in range(len(pivot_wr.index)):
    for j in range(len(pivot_wr.columns)):
        val = pivot_wr.values[i, j]
        n = int(pivot_n.values[i, j])
        if not np.isnan(val) and n > 10:
            color = 'black' if val > 75 else 'white'
            ax.text(j, i, f'{val:.0f}%\nn={n}', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, label='Win Rate %')
plt.tight_layout()
plt.show()

print("\nKey insight: The bottom-left corner (Tiny + Age 1) is where almost all losses live.")
print("The question is: can we filter losses WITHOUT just skipping those cells?")

---

## 4. Exploration Lab — New Ideas

Goal: find filters that **increase accuracy but skip less than v2.5's 10%**.

The existing strategies only use two signals: **move size** and **momentum age**. What else can we extract from 5-second candle data?

### Idea A: Intra-Minute Volatility

Instead of just the move (close-to-close), look at the **range** within the minute. A tiny close-to-close move with a wide high-low range means the price bounced around — indecisive. A tiny move with a narrow range means genuinely flat.

In [ ]:
# Build 1-minute OHLC bars from 5-second candles
df['minute_group'] = df.index.floor('min')
minute_bars = df.groupby('minute_group').agg(
    open=('open', 'first'),
    high=('high', 'max'),
    low=('low', 'min'),
    close=('close', 'last'),
    n_candles=('close', 'count'),
).dropna()

minute_bars['range'] = minute_bars['high'] - minute_bars['low']
minute_bars['body'] = abs(minute_bars['close'] - minute_bars['open'])
minute_bars['wick_ratio'] = (minute_bars['range'] - minute_bars['body']) / minute_bars['range']
minute_bars['wick_ratio'] = minute_bars['wick_ratio'].fillna(0)

# Merge into candles_00
candles_00['minute_key'] = candles_00.index.floor('min')
candles_00 = candles_00.join(minute_bars[['range', 'body', 'wick_ratio']], on='minute_key', rsuffix='_bar')

# Use PREVIOUS minute's bar features for prediction
candles_00['prev_range'] = candles_00['range'].shift(1)
candles_00['prev_body'] = candles_00['body'].shift(1)
candles_00['prev_wick_ratio'] = candles_00['wick_ratio'].shift(1)
candles_00 = candles_00.dropna(subset=['prev_range'])

# Win rate by wick ratio (how much of the bar is wicks vs body)
candles_00['wick_bucket'] = pd.cut(candles_00['prev_wick_ratio'], 
    bins=[0, 0.3, 0.5, 0.7, 1.0], labels=['Low (<30%)', 'Med (30-50%)', 'High (50-70%)', 'VHigh (>70%)'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

by_wick = candles_00.groupby('wick_bucket')['follow_last_correct'].agg(['mean', 'count'])
colors = ['lime' if m > 0.87 else 'yellow' if m > 0.80 else 'red' for m in by_wick['mean']]
bars = axes[0].bar(range(len(by_wick)), by_wick['mean'] * 100, color=colors)
axes[0].set_xticks(range(len(by_wick)))
axes[0].set_xticklabels(by_wick.index)
axes[0].axhline(y=86.9, color='cyan', linestyle='--', alpha=0.5, label='v2 baseline')
axes[0].set_title('Win Rate by Previous Minute Wick Ratio')
axes[0].set_ylabel('Win Rate (%)')
axes[0].set_ylim(60, 100)
axes[0].legend()
for bar, (_, row) in zip(bars, by_wick.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'n={int(row["count"])}', ha='center', fontsize=9, color='white')

# Win rate by range size
candles_00['range_bucket'] = pd.qcut(candles_00['prev_range'], q=5, labels=['p0-20', 'p20-40', 'p40-60', 'p60-80', 'p80-100'])
by_range = candles_00.groupby('range_bucket')['follow_last_correct'].agg(['mean', 'count'])
colors = ['lime' if m > 0.87 else 'yellow' if m > 0.80 else 'red' for m in by_range['mean']]
bars = axes[1].bar(range(len(by_range)), by_range['mean'] * 100, color=colors)
axes[1].set_xticks(range(len(by_range)))
axes[1].set_xticklabels(by_range.index)
axes[1].axhline(y=86.9, color='cyan', linestyle='--', alpha=0.5, label='v2 baseline')
axes[1].set_title('Win Rate by Previous Minute Range (quintiles)')
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_ylim(60, 100)
axes[1].legend()
for bar, (_, row) in zip(bars, by_range.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'n={int(row["count"])}', ha='center', fontsize=9, color='white')

plt.tight_layout()
plt.show()

# Backtest: skip only high-wick minutes (>70% wick = indecisive)
mask_wick = candles_00['prev_wick_ratio'] <= 0.7
mask_wick.iloc[0] = True
r_wick = backtest(candles_00, mask_wick)
print("\nBacktest — skip high-wick (>70%) minutes:")
print_result("Idea A: skip wick>70%", r_wick)
print_result("v2.5 (reference)", r_v25)

### Idea B: Last 10-Second Momentum (Pre-Entry Confirmation)

Check the price direction in the last 10 seconds before entry (:50 to :00). If it contradicts the predicted direction, the trend might be breaking.

In [ ]:
# Get price at :50 of each minute for pre-entry momentum
candles_50 = df[df['second'] == 50][['close']].copy()
candles_50 = candles_50[~candles_50.index.duplicated(keep='first')]
candles_50.columns = ['price_at_50']
candles_50['minute_key'] = candles_50.index.floor('min')

# Merge: for each :00 entry, get the :50 price from the PREVIOUS minute
candles_00['prev_minute_key'] = candles_00['minute_key'] - pd.Timedelta(minutes=1)
candles_00 = candles_00.merge(
    candles_50[['minute_key', 'price_at_50']].rename(columns={'minute_key': 'prev_minute_key'}),
    on='prev_minute_key', how='left'
)
candles_00.index = candles_00['minute_key']  # restore index

# Last 10s direction: did price go up from :50 to :00?
candles_00['last_10s_up'] = (candles_00['entry_price'] > candles_00['price_at_50']).astype(float)
candles_00['last_10s_up'] = candles_00['last_10s_up'].fillna(0.5)

# Does the last 10s confirm or contradict the predicted direction?
candles_00['confirms'] = (candles_00['last_10s_up'] == candles_00['prev_went_up']).astype(int)

wr_confirms = candles_00[candles_00['confirms'] == 1]['follow_last_correct'].mean()
wr_contradicts = candles_00[candles_00['confirms'] == 0]['follow_last_correct'].mean()
n_confirms = (candles_00['confirms'] == 1).sum()
n_contradicts = (candles_00['confirms'] == 0).sum()

print(f"Last 10s CONFIRMS trend:     WR = {wr_confirms:.1%}  (n={n_confirms})")
print(f"Last 10s CONTRADICTS trend:  WR = {wr_contradicts:.1%}  (n={n_contradicts})")
print(f"Contradicts = {n_contradicts / len(candles_00) * 100:.1f}% of trades")

# Backtest: skip when last 10s contradicts
mask_confirm = candles_00['confirms'] == 1
mask_confirm.iloc[0] = True
r_confirm = backtest(candles_00, mask_confirm)
print(f"\nBacktest — skip when last 10s contradicts:")
print_result("Idea B: skip contradictions", r_confirm)
print_result("v2.5 (reference)", r_v25)

### Idea C: Acceleration (Move Getting Stronger or Weaker?)

If the current move is LARGER than the previous move, momentum is accelerating — stronger signal. If it's shrinking, momentum may be fading.

In [ ]:
# Acceleration: is the move getting bigger or smaller?
candles_00['prev_prev_move'] = candles_00['prev_move'].shift(1)
candles_00['acceleration'] = candles_00['prev_move'] - candles_00['prev_prev_move']
candles_00['accelerating'] = (candles_00['acceleration'] > 0).astype(int)

valid = candles_00.dropna(subset=['acceleration'])

wr_accel = valid[valid['accelerating'] == 1]['follow_last_correct'].mean()
wr_decel = valid[valid['accelerating'] == 0]['follow_last_correct'].mean()
n_accel = (valid['accelerating'] == 1).sum()
n_decel = (valid['accelerating'] == 0).sum()

print(f"Accelerating (move growing):  WR = {wr_accel:.1%}  (n={n_accel}, {n_accel/len(valid)*100:.0f}%)")
print(f"Decelerating (move shrinking): WR = {wr_decel:.1%}  (n={n_decel}, {n_decel/len(valid)*100:.0f}%)")

# Backtest: skip decelerating
mask_accel = valid['accelerating'] == 1
mask_accel.iloc[0] = True
r_accel = backtest(valid, mask_accel)
print(f"\nBacktest — skip decelerating moves:")
print_result("Idea C: accel only", r_accel)
print_result("v2.5 (reference)", r_v25)

### Idea D: Consecutive Direction Count (Trend Length)

Instead of using momentum age as a binary filter (age >= 2), use it as a continuous signal. Longer trends are more stable — but very long trends might be about to reverse.

In [ ]:
# Win rate by exact momentum age (1, 2, 3, ... 20+)
max_age_show = 20
age_wr = []
for age in range(1, max_age_show + 1):
    if age < max_age_show:
        subset = candles_00[candles_00['prev_momentum_age'] == age]
    else:
        subset = candles_00[candles_00['prev_momentum_age'] >= age]
    if len(subset) > 20:
        age_wr.append({
            'age': f'{age}' if age < max_age_show else f'{age}+',
            'win_rate': subset['follow_last_correct'].mean(),
            'count': len(subset),
            'pct_of_trades': len(subset) / len(candles_00) * 100,
        })

age_df = pd.DataFrame(age_wr)

fig, ax1 = plt.subplots(figsize=(14, 5))
colors = ['red' if wr < 0.80 else 'yellow' if wr < 0.90 else 'lime' for wr in age_df['win_rate']]
bars = ax1.bar(range(len(age_df)), age_df['win_rate'] * 100, color=colors, alpha=0.8)
ax1.set_xticks(range(len(age_df)))
ax1.set_xticklabels(age_df['age'])
ax1.axhline(y=86.9, color='cyan', linestyle='--', alpha=0.5, label='v2 baseline')
ax1.set_xlabel('Momentum Age (consecutive same-direction minutes)')
ax1.set_ylabel('Win Rate (%)')
ax1.set_ylim(60, 100)
ax1.set_title('Win Rate by Exact Momentum Age')
ax1.legend(loc='lower right')

ax2 = ax1.twinx()
ax2.plot(range(len(age_df)), age_df['pct_of_trades'], 'o-', color='white', alpha=0.5, markersize=4)
ax2.set_ylabel('% of all trades', color='white', alpha=0.5)

for bar, (_, row) in zip(bars, age_df.iterrows()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'n={int(row["count"])}', ha='center', fontsize=7, color='white')

plt.tight_layout()
plt.show()

# Key finding
age1_pct = candles_00[candles_00['prev_momentum_age'] == 1].shape[0] / len(candles_00) * 100
print(f"\nAge=1 is {age1_pct:.0f}% of trades — skipping it removes a lot of opportunities.")
print("But the win rate jump from age 1→2 is the biggest single improvement.")

### Idea E: Bet AGAINST the Trend at Age=1

The biggest loss bucket is age=1 (fresh reversals). Instead of skipping these, what if we **fade** them — bet that the previous trend continues instead of following the new reversal?

If the trend was UP for 5 minutes and just flipped DOWN (age=1), maybe bet UP (the old trend) instead of DOWN (the new signal).

In [ ]:
# For age=1, what if we bet the OPPOSITE of follow-last (i.e., fade the reversal)?
age1 = candles_00[candles_00['prev_momentum_age'] == 1]
age1_follow = age1['follow_last_correct'].mean()
age1_fade = 1 - age1_follow  # fading = opposite of following

print(f"Age=1 trades: {len(age1)} ({len(age1)/len(candles_00)*100:.0f}% of all)")
print(f"  Follow last (normal):  {age1_follow:.1%}")
print(f"  Fade (bet opposite):   {age1_fade:.1%}")
print(f"\n  {'Fading is BETTER' if age1_fade > age1_follow else 'Following is still better'}")

# What about age=1 with tiny moves specifically?
age1_tiny = candles_00[(candles_00['prev_momentum_age'] == 1) & (candles_00['prev_move'] < 0.00019)]
age1_tiny_follow = age1_tiny['follow_last_correct'].mean()
age1_tiny_fade = 1 - age1_tiny_follow

print(f"\nAge=1 + Tiny move: {len(age1_tiny)} trades")
print(f"  Follow: {age1_tiny_follow:.1%}")
print(f"  Fade:   {age1_tiny_fade:.1%}")

# Full backtest: follow for age>=2, fade for age=1
def backtest_with_fade(trades_df, fade_mask, payout=0.85, max_losses=8, max_stake=200, base_stake=1.0):
    """Like backtest, but FADE (bet opposite) for trades where fade_mask is True."""
    outcomes = trades_df['went_up'].values
    fade = fade_mask.values if hasattr(fade_mask, 'values') else np.array(fade_mask)
    
    balance = 0.0
    equity = [0.0]
    stake = base_stake
    consec_losses = 0
    wins = 0; losses = 0; busts = 0
    max_drawdown = 0.0; peak = 0.0
    loss_streaks = []; current_streak = 0
    last_direction = None
    
    for i in range(len(outcomes)):
        if last_direction is None:
            bet = random.randint(0, 1)
        else:
            bet = last_direction if not fade[i] else (1 - last_direction)
        
        if consec_losses >= max_losses or stake > max_stake:
            busts += 1; stake = base_stake; consec_losses = 0
        
        won = (bet == outcomes[i])
        if won:
            balance += stake * payout; wins += 1; stake = base_stake; consec_losses = 0
            if current_streak > 0: loss_streaks.append(current_streak)
            current_streak = 0
        else:
            balance -= stake; losses += 1; consec_losses += 1; current_streak += 1
            stake = round((stake + base_stake) / payout, 2)
        
        if balance > peak: peak = balance
        dd = peak - balance
        if dd > max_drawdown: max_drawdown = dd
        last_direction = outcomes[i]
        equity.append(balance)
    
    if current_streak > 0: loss_streaks.append(current_streak)
    total = wins + losses
    hours_traded = total / 60
    return {
        'wins': wins, 'losses': losses, 'skips': 0,
        'total_trades': total, 'total_opportunities': total,
        'win_rate': wins / total if total > 0 else 0,
        'skip_pct': 0,
        'profit': balance,
        'per_trade': balance / total if total > 0 else 0,
        'per_hour': balance / hours_traded if hours_traded > 0 else 0,
        'busts': busts, 'max_drawdown': max_drawdown,
        'max_loss_streak': max(loss_streaks) if loss_streaks else 0,
        'equity': equity,
    }

# Fade at age=1 only
fade_mask = candles_00['prev_momentum_age'] == 1
r_fade = backtest_with_fade(candles_00, fade_mask)
print(f"\nBacktest — Fade at age=1, follow otherwise (0% skips!):")
print_result("Idea E: fade age=1", r_fade)
print_result("v2 (reference)", r_v2)
print_result("v2.5 (reference)", r_v25)

### Idea F: Time-of-Day Filter

Does the OTC algorithm behave differently at different hours? If certain hours have notably lower win rates, we could skip or adjust during those periods.

In [ ]:
# Win rate by hour of day (UTC)
candles_00['hour'] = candles_00.index.hour
by_hour = candles_00.groupby('hour')['follow_last_correct'].agg(['mean', 'count'])

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['red' if m < 0.82 else 'yellow' if m < 0.87 else 'lime' for m in by_hour['mean']]
bars = ax.bar(by_hour.index, by_hour['mean'] * 100, color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(y=86.9, color='cyan', linestyle='--', alpha=0.5, label='v2 baseline (86.9%)')
ax.axhline(y=54.05, color='yellow', linestyle='--', alpha=0.3, label='break-even')
ax.set_xlabel('Hour (UTC)')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Win Rate by Hour of Day')
ax.set_ylim(70, 100)
ax.legend()

for bar, (hour, row) in zip(bars, by_hour.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
           f'{row["mean"]*100:.1f}%\nn={int(row["count"])}', ha='center', fontsize=7, color='white')

plt.tight_layout()
plt.show()

# Find worst hours
worst = by_hour[by_hour['mean'] < 0.85].sort_values('mean')
if len(worst) > 0:
    print("Hours below 85% win rate:")
    for hour, row in worst.iterrows():
        print(f"  {hour:02d}:00 UTC — {row['mean']:.1%} ({int(row['count'])} trades)")
    
    # Backtest: skip worst hours
    bad_hours = worst.index.tolist()
    mask_hours = ~candles_00['hour'].isin(bad_hours)
    mask_hours.iloc[0] = True
    r_hours = backtest(candles_00, mask_hours)
    print(f"\nBacktest — skip hours {bad_hours}:")
    print_result(f"Idea F: skip {len(bad_hours)} worst hours", r_hours)
    print_result("v2.5 (reference)", r_v25)
else:
    print("All hours are above 85% — no clear time-of-day edge.")

### Idea G: Rolling Volatility Regime

Split the data into "calm" vs "volatile" regimes based on rolling average move size. High-volatility periods might have stronger momentum (bigger moves = more signal).

In [ ]:
# Rolling volatility: average move size over last N minutes
for window in [5, 10, 20]:
    candles_00[f'vol_{window}'] = candles_00['move'].rolling(window).mean().shift(1)

candles_00_vol = candles_00.dropna(subset=['vol_10'])

# Split into volatility quintiles
candles_00_vol['vol_quintile'] = pd.qcut(candles_00_vol['vol_10'], q=5, 
    labels=['Very Calm', 'Calm', 'Normal', 'Active', 'Very Active'])

by_vol = candles_00_vol.groupby('vol_quintile')['follow_last_correct'].agg(['mean', 'count'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['red' if m < 0.82 else 'yellow' if m < 0.87 else 'lime' for m in by_vol['mean']]
bars = axes[0].bar(range(len(by_vol)), by_vol['mean'] * 100, color=colors)
axes[0].set_xticks(range(len(by_vol)))
axes[0].set_xticklabels(by_vol.index, rotation=15)
axes[0].axhline(y=86.9, color='cyan', linestyle='--', alpha=0.5, label='v2 baseline')
axes[0].set_title('Win Rate by Volatility Regime (10-min rolling avg)')
axes[0].set_ylabel('Win Rate (%)')
axes[0].set_ylim(70, 100)
axes[0].legend()
for bar, (_, row) in zip(bars, by_vol.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{row["mean"]*100:.1f}%\nn={int(row["count"])}', ha='center', fontsize=8, color='white')

# Scatter: rolling volatility vs rolling win rate
roll_wr = candles_00_vol['follow_last_correct'].rolling(100).mean()
roll_vol = candles_00_vol['vol_10']
axes[1].scatter(roll_vol * 10000, roll_wr * 100, s=1, alpha=0.3, color='cyan')
axes[1].set_xlabel('Rolling Volatility (pips × 10⁴)')
axes[1].set_ylabel('Rolling Win Rate (100-trade) (%)')
axes[1].set_title('Volatility vs Win Rate')
axes[1].axhline(y=86.9, color='lime', linestyle='--', alpha=0.5)
axes[1].set_ylim(50, 100)

plt.tight_layout()
plt.show()

# Backtest: skip "Very Calm" periods
mask_vol = candles_00_vol['vol_quintile'] != 'Very Calm'
mask_vol.iloc[0] = True
r_vol = backtest(candles_00_vol, mask_vol)
print("Backtest — skip 'Very Calm' volatility regime:")
print_result("Idea G: skip very calm", r_vol)
print_result("v2.5 (reference)", r_v25)

### Idea H: Adaptive Skip — Only Skip Tiny Moves That Are ALSO Fresh Reversals

v2.5 skips ALL tiny moves. v3 skips tiny OR fresh. What if we only skip trades that are BOTH tiny AND fresh? This is more surgical — it targets the exact overlap where losses concentrate.

In [ ]:
# Surgical skip: only skip when BOTH tiny AND age=1
mask_surgical = ~((candles_00['prev_move'] < 0.00019) & (candles_00['prev_momentum_age'] == 1))
mask_surgical.iloc[0] = True
r_surgical = backtest(candles_00, mask_surgical)

# Compare: skip tiny only, skip fresh only, skip both (OR), skip both (AND)
mask_tiny_only = candles_00['prev_move'] >= 0.00019
mask_tiny_only.iloc[0] = True
r_tiny_only = backtest(candles_00, mask_tiny_only)

mask_fresh_only = candles_00['prev_momentum_age'] >= 2
mask_fresh_only.iloc[0] = True
r_fresh_only = backtest(candles_00, mask_fresh_only)

mask_or = (candles_00['prev_move'] >= 0.00019) & (candles_00['prev_momentum_age'] >= 2)
mask_or.iloc[0] = True
r_or = backtest(candles_00, mask_or)

print("Skip filter comparison:")
print(f"{'Strategy':<45} | {'WR':>5} | {'Skip%':>5} | {'Profit':>10} | {'$/hr':>7}")
print("—" * 85)
print_result("v2 (no filter)", r_v2)
print_result("Skip tiny only (v2.5)", r_tiny_only)
print_result("Skip fresh only (age<2)", r_fresh_only)
print_result("Skip tiny AND fresh (surgical)", r_surgical)
print_result("Skip tiny OR fresh (v3)", r_or)

# Visualize what gets skipped
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
skip_types = {
    'v2.5 (tiny)': ~mask_tiny_only,
    'Surgical (tiny+fresh)': ~mask_surgical,
    'v3 (tiny OR fresh)': ~mask_or,
}
for ax, (name, skip) in zip(axes, skip_types.items()):
    skipped = candles_00[skip]
    traded = candles_00[~skip]
    skip_pct = skip.sum() / len(candles_00) * 100
    skip_wr = skipped['follow_last_correct'].mean() * 100 if len(skipped) > 0 else 0
    trade_wr = traded['follow_last_correct'].mean() * 100
    
    ax.bar([0, 1], [trade_wr, skip_wr], color=['lime', 'red'], width=0.6)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([f'Traded\n({100-skip_pct:.0f}%)', f'Skipped\n({skip_pct:.0f}%)'])
    ax.set_ylabel('Win Rate (%)')
    ax.set_title(name)
    ax.set_ylim(50, 100)
    ax.axhline(y=86.9, color='cyan', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

### Idea I: Combo Strategies

Mix the best ideas together. Try combinations that maximize $/hour while keeping skip% low.

In [ ]:
# Test many combinations — grid search over filters
combos = []

# Move thresholds to try
move_thresholds = [0, 0.00010, 0.00015, 0.00019, 0.00025, 0.00030]
# Age thresholds
age_thresholds = [1, 2, 3]  # skip if age < threshold
# Logic: AND (both required) vs OR (either triggers skip)
logic_options = ['and', 'or']

for move_t in move_thresholds:
    for age_t in age_thresholds:
        for logic in logic_options:
            if logic == 'and':
                # Skip only if BOTH conditions met
                mask = ~((candles_00['prev_move'] < move_t) & (candles_00['prev_momentum_age'] < age_t))
            else:
                # Skip if EITHER condition met
                mask = (candles_00['prev_move'] >= move_t) & (candles_00['prev_momentum_age'] >= age_t)
            
            if move_t == 0 and age_t == 1:
                continue  # no filter = v2
            
            mask.iloc[0] = True
            r = backtest(candles_00, mask)
            combos.append({
                'move_threshold': move_t,
                'age_threshold': age_t,
                'logic': logic,
                'win_rate': r['win_rate'],
                'skip_pct': r['skip_pct'],
                'profit': r['profit'],
                'per_hour': r['per_hour'],
                'busts': r['busts'],
                'max_loss_streak': r['max_loss_streak'],
            })

combos_df = pd.DataFrame(combos)
combos_df = combos_df.sort_values('per_hour', ascending=False)

# Show top 15 by $/hour
print("Top 15 combos by $/hour:")
print(f"{'Move':>8} {'Age':>4} {'Logic':>5} | {'WR':>5} | {'Skip':>5} | {'$/hr':>7} | {'Profit':>8} | {'Busts':>5} | {'Streak':>6}")
print("—" * 75)
for _, row in combos_df.head(15).iterrows():
    print(f"{row['move_threshold']:.5f} {int(row['age_threshold']):>4} {row['logic']:>5} | "
          f"{row['win_rate']:>5.1%} | {row['skip_pct']:>5.1%} | "
          f"${row['per_hour']:>6.2f} | ${row['profit']:>7.2f} | "
          f"{int(row['busts']):>5} | {int(row['max_loss_streak']):>5}L")

In [ ]:
# Pareto frontier: plot all combos as WR vs $/hour, sized by skip%
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# All combos
sc = axes[0].scatter(combos_df['win_rate'] * 100, combos_df['per_hour'], 
                     c=combos_df['skip_pct'] * 100, cmap='RdYlGn_r', 
                     s=80, alpha=0.7, edgecolors='white', linewidth=0.5)
plt.colorbar(sc, ax=axes[0], label='Skip %')
axes[0].set_xlabel('Win Rate (%)')
axes[0].set_ylabel('$/hour')
axes[0].set_title('All Filter Combos — Win Rate vs $/hour\n(color = skip %)')
axes[0].grid(alpha=0.2)

# Highlight the existing strategies
for name, r, marker, color in [
    ('v2', r_v2, '*', 'lime'),
    ('v2.5', r_v25, '*', 'cyan'),
    ('v3', r_v3, '*', 'orange'),
    ('v4', r_v4, '*', 'magenta'),
]:
    axes[0].scatter(r['win_rate'] * 100, r['per_hour'], s=300, marker=marker, 
                   color=color, edgecolors='white', linewidth=2, zorder=10)
    axes[0].annotate(name, (r['win_rate'] * 100, r['per_hour']),
                    textcoords="offset points", xytext=(10, 5), fontsize=10, color=color, fontweight='bold')

# Pareto frontier: skip% vs $/hour
axes[1].scatter(combos_df['skip_pct'] * 100, combos_df['per_hour'],
               c=combos_df['win_rate'] * 100, cmap='RdYlGn', 
               s=80, alpha=0.7, edgecolors='white', linewidth=0.5)
plt.colorbar(sc, ax=axes[1], label='Win Rate %')
axes[1].set_xlabel('Skip %')
axes[1].set_ylabel('$/hour')
axes[1].set_title('Skip % vs $/hour\n(less skip = more trades = potentially more profit)')
axes[1].grid(alpha=0.2)

for name, r, color in [('v2', r_v2, 'lime'), ('v2.5', r_v25, 'cyan'), ('v3', r_v3, 'orange')]:
    axes[1].scatter(r['skip_pct'] * 100, r['per_hour'], s=300, marker='*',
                   color=color, edgecolors='white', linewidth=2, zorder=10)
    axes[1].annotate(name, (r['skip_pct'] * 100, r['per_hour']),
                    textcoords="offset points", xytext=(10, 5), fontsize=10, color=color, fontweight='bold')

plt.tight_layout()
plt.show()

# Best combo with <=5% skip
best_low_skip = combos_df[combos_df['skip_pct'] <= 0.05].head(1)
if len(best_low_skip) > 0:
    b = best_low_skip.iloc[0]
    print(f"\nBest combo with <=5% skip:")
    print(f"  Move >= {b['move_threshold']:.5f}, Age >= {int(b['age_threshold'])}, Logic: {b['logic']}")
    print(f"  WR: {b['win_rate']:.1%} | Skip: {b['skip_pct']:.1%} | $/hr: ${b['per_hour']:.2f}")

# Best combo with <=10% skip
best_med_skip = combos_df[combos_df['skip_pct'] <= 0.10].head(1)
if len(best_med_skip) > 0:
    b = best_med_skip.iloc[0]
    print(f"\nBest combo with <=10% skip:")
    print(f"  Move >= {b['move_threshold']:.5f}, Age >= {int(b['age_threshold'])}, Logic: {b['logic']}")
    print(f"  WR: {b['win_rate']:.1%} | Skip: {b['skip_pct']:.1%} | $/hr: ${b['per_hour']:.2f}")

---

## 5. Summary — What Works?

Run this cell after exploring ideas above to see all results side by side.

In [ ]:
# Final comparison — all strategies and ideas
all_results = {
    'v1 (:30, no filter)': r_v1,
    'v2 (:00, no filter)': r_v2,
    'v2.5 (skip tiny)': r_v25,
    'v3 (skip tiny OR fresh)': r_v3,
    'v4 (large only)': r_v4,
}

# Add exploration results (check if they exist)
idea_results = {}
if 'r_wick' in dir(): idea_results['A: skip high wick'] = r_wick
if 'r_confirm' in dir(): idea_results['B: skip contradictions'] = r_confirm
if 'r_accel' in dir(): idea_results['C: accel only'] = r_accel
if 'r_fade' in dir(): idea_results['E: fade age=1'] = r_fade
if 'r_hours' in dir(): idea_results['F: skip bad hours'] = r_hours
if 'r_vol' in dir(): idea_results['G: skip very calm'] = r_vol
if 'r_surgical' in dir(): idea_results['H: surgical (tiny+fresh)'] = r_surgical

print("=" * 120)
print(f"{'STRATEGY':<45} | {'WR':>5} | {'Skip%':>5} | {'Profit':>10} | {'$/hr':>7} | {'Busts':>5} | {'MaxStreak':>9}")
print("=" * 120)

print("\n— Existing Strategies —")
for name, r in all_results.items():
    print_result(name, r)

print("\n— New Ideas —")
for name, r in idea_results.items():
    print_result(name, r)

# Rank by $/hour
print("\n\n— RANKED BY $/HOUR —")
combined = {**all_results, **idea_results}
ranked = sorted(combined.items(), key=lambda x: x[1]['per_hour'], reverse=True)
for i, (name, r) in enumerate(ranked, 1):
    marker = " ★" if r['per_hour'] == ranked[0][1]['per_hour'] else ""
    print(f"  {i}. ", end="")
    print_result(name, r)
    if marker:
        print(f"     {marker}")

## 6. Your Turn — Custom Filter

Use this cell to test your own filter ideas. Set `my_mask` to a boolean Series — `True` = trade, `False` = skip.

In [ ]:
# Available columns in candles_00:
# - prev_move: absolute price change of previous trade
# - prev_went_up: 1 if previous trade went up, 0 if down
# - prev_momentum_age: how many consecutive same-direction results
# - prev_range: high-low range of previous minute bar
# - prev_body: body size (abs(close-open)) of previous minute bar  
# - prev_wick_ratio: wick/range ratio of previous minute bar
# - confirms: 1 if last 10s direction matches predicted direction
# - acceleration: move change (current move - previous move)
# - vol_5, vol_10, vol_20: rolling average move size (5/10/20 min)
# - hour: hour of day (UTC)

# ===== EDIT YOUR FILTER HERE =====
my_mask = (
    (candles_00['prev_move'] >= 0.00015)  # example: slightly less aggressive than v2.5
    # & (candles_00['prev_momentum_age'] >= 2)  # uncomment to add age filter
    # & (candles_00['confirms'] == 1)            # uncomment to add confirmation
)
my_mask.iloc[0] = True

r_custom = backtest(candles_00, my_mask)
print("Your custom strategy:")
print_result("Custom", r_custom)
print()
print_result("v2 (baseline)", r_v2)
print_result("v2.5 (reference)", r_v25)